In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder,StandardScaler,MinMaxScaler,LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path + '/Q1_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df["Delivery_Time"].head()

In [ ]:
# Task 5: Write your code here:
plt.hist(df["Delivery_Time"],bins=30)

In [ ]:
df.head()

In [ ]:
# Task 1: Write your code here:
df.drop("Order_ID",axis=1,inplace=True)

In [ ]:
# Task 2: Write your code here:
df.isna().sum()

In [ ]:
missing_cols = df.columns.drop(["Distance_km","Vehicle_Type","Preparation_Time_min"])
df[missing_cols]

In [ ]:
#Not good to fill Target Variable's values, so better to drop them
df.dropna(subset="Delivery_Time",inplace=True)

In [ ]:
df["Courier_Experience_yrs"].unique()

In [ ]:
df["Courier_Experience_yrs"].fillna(round(df["Courier_Experience_yrs"].mean()),inplace=True)

In [ ]:
df["Time_of_Day"].unique()

In [ ]:
df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0],inplace=True)

In [ ]:
df["Traffic_Level"].unique()

In [ ]:
df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0],inplace=True)

In [ ]:
df["Weather"].unique()

In [ ]:
df["Weather"].fillna(df["Weather"].mode()[0],inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
# Task 3: Write your code here:
print(df.duplicated().sum())
if df.duplicated().sum() > 0:
  df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.head()

In [ ]:
# Task 4: Write your code here:
df.select_dtypes(include=["object"]).columns

In [ ]:
print(df["Weather"].unique())
print(df["Traffic_Level"].unique())
print(df["Time_of_Day"].unique())
print(df["Vehicle_Type"].unique())

In [ ]:
cat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
ohe = OneHotEncoder()
ohe_result = ohe.fit_transform(df[['Weather', 'Time_of_Day', 'Vehicle_Type']])

In [ ]:
temp_df = pd.DataFrame(columns=ohe.get_feature_names_out(),data=ohe_result.toarray())
temp_df.head()

In [ ]:
dfOHE = pd.concat([df.drop(['Weather', 'Time_of_Day', 'Vehicle_Type'],axis=1),temp_df],axis=1)
dfOHE.head()

In [ ]:
df.shape

In [ ]:
dfOHE.shape

In [ ]:
dfOHE.dropna(inplace=True)

In [ ]:
le = LabelEncoder()
dfOHE["Traffic_Level"] = le.fit_transform(dfOHE["Traffic_Level"])

In [ ]:
dfOHE.head()

In [ ]:
# Task 5: Write your code here:
dcols = dfOHE.columns.drop("Delivery_Time") ## desired columns (not scaling target)
sc = StandardScaler()
dfOHE[dcols] = sc.fit_transform(dfOHE[dcols])

In [ ]:
# Task 6: Write your code here:
dfOHE["Delivery_Time"].skew()

In [ ]:
plt.hist(dfOHE["Delivery_Time"],bins=50)

In [ ]:
dfOHE.head()

In [ ]:
# Task 1: Write your code here:
X = dfOHE.drop("Delivery_Time",axis=1).values
y = dfOHE["Delivery_Time"].values


In [ ]:
# Task 2,3,4,5: Write your code here:
kf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
model=RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42)
mae_scores = []
predictions = []

In [ ]:
for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), start=1):
    X_fold_train, X_fold_val = X[train_idx], X[test_idx]
    y_fold_train, y_fold_val = y[train_idx], y[test_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)
    predictions.append(y_fold_pred)
    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores).mean()
print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores}")

In [ ]:
len(model.feature_importances_)

In [ ]:
len(dfOHE.columns.drop("Delivery_Time"))

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': dfOHE.columns.drop("Delivery_Time"),
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(predictions,bins=30)

In [ ]:
# Task Bonus: Write your code here: